# Add SSH and wind products as features


Global Ocean Gridded L 4 Sea Surface Heights And Derived Variables Reprocessed 1993 Ongoing

https://data.marine.copernicus.eu/product/SEALEVEL_GLO_PHY_L4_MY_008_047/description



- Variable abbrevations (https://www.cmcc.it/wp-content/uploads/2021/03/CMEMS-MED-PUM-006-013-V1.1.pdf)

In [ ]:
# To download CMEMS SSH, ran from subset_ssh.ipynb in the <copernicusmarine> folder
# ~15 min run time for 10 years


# import copernicusmarine
# yrs = range(2015, 2023)

# for yr in yrs:
#     print('==> Downloading year ' + str(yr))
#     copernicusmarine.subset(
#     dataset_id = 'cmems_obs-sl_glo_phy-ssh_my_allsat-l4-duacs-0.125deg_P1D',
#     variables = ['adt', 'sla'],
#     start_datetime = str(yr) + '-01-01',
#     end_datetime = str(yr) + '-12-31', 
#     minimum_latitude = -89,
#     maximum_latitude = -35
#     )

# import xarray as xr
# annual_dict = {k:None for k in range(2014,2024)}
# for open_yr in annual_dict.keys():
#     filepath = ('cmems_obs-sl_glo_phy-ssh_my_allsat-l4-duacs-0.125deg_P1D_adt-sla_179.94W-179.94E_88.94S-35.06S_' + str(open_yr) + '-01-01-' + str(open_yr) + '-12-31.nc')
#     annual_dict[open_yr] = xr.open_dataset(filepath)


In [27]:
import mod_loading as loader
from importlib import reload
import numpy as np
import pandas as pd
import xarray as xr
import datetime

import gsw
import mod_argo
import mod_ocean

import matplotlib.pyplot as plt
from cmocean import cm as cmo

import mod_regression as myreg
reload(myreg)

<module 'mod_regression' from '/Users/sangminsong/Library/CloudStorage/OneDrive-UW/Code/CRUSOE/src/mod_regression.py'>

In [28]:
from tqdm import tqdm

# Setup data

In [39]:
reload(loader)
[floatDF, shipDF] = loader.import_regression_inputs()

allplat =  pd.concat([floatDF, shipDF], axis=0)
allplat =  mod_ocean.expand_datetime(allplat, type='dataframe')


In [30]:
import xarray as xr

adt_dict = {k:None for k in range(2014,2024)}
for open_yr in adt_dict.keys():
    folder = '/Users/sangminsong/Library/CloudStorage/OneDrive-UW/Code/CRUSOE/copernicusmarine/'
    filepath = ('cmems_obs-sl_glo_phy-ssh_my_allsat-l4-duacs-0.125deg_P1D_adt-sla_179.94W-179.94E_88.94S-35.06S_' + str(open_yr) + '-01-01-' + str(open_yr) + '-12-31.nc')
    adt_dict[open_yr] = xr.open_dataset(folder + filepath)

In [ ]:
# # Try for one year                   
# yr=2014

# plat_data = allplat[allplat.year==yr]
# adt_data = adt_dict[yr]
# # plat_data['adt'] = np.tile(np.nan, len(plat_data))

# def get_adt_for_obs(row):
#     return adt_data.sel(time=row.datetime, latitude=row.latitude, longitude=row.longitude, method='nearest').adt.values.tolist()
# def get_sla_for_obs(row):
#     return adt_data.sel(time=row.datetime, latitude=row.latitude, longitude=row.longitude, method='nearest').sla.values.tolist()


# # plat_data['adt'] = plat_data.apply(lambda row: adt_data.sel(time=row.datetime, latitude=row.latitude, longitude=row.longitude, method='nearest'), axis=1)
# plat_data['adt'] = plat_data.apply(lambda row: get_adt_for_obs(row), axis=1)
# plat_data['sla'] = plat_data.apply(lambda row: get_sla_for_obs(row), axis=1)
# # 



/var/folders/nt/sjynqxjj7cz9r15fkd5d4r_40000gp/T/ipykernel_93258/3857722186.py:16: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  plat_data['adt'] = plat_data.apply(lambda row: get_adt_for_obs(row), axis=1)
/var/folders/nt/sjynqxjj7cz9r15fkd5d4r_40000gp/T/ipykernel_93258/3857722186.py:17: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  plat_data['sla'] = plat_data.apply(lambda row: get_sla_for_obs(row), axis=1)


In [40]:
# Run for all ten years

allplat_annual = {k:None for k in range(2014,2024)}

def get_adt_for_obs(row):
    return adt_data.sel(time=row.datetime, latitude=row.latitude, longitude=row.longitude, method='nearest').adt.values.tolist()
def get_sla_for_obs(row):
    return adt_data.sel(time=row.datetime, latitude=row.latitude, longitude=row.longitude, method='nearest').sla.values.tolist()


# for yr in range(2014,2024):
#     allplat_annual[yr] = allplat[allplat.year==yr]

for yr in range(2014,2024):
    allplat_annual[yr] = allplat[allplat.year==yr]
    adt_data = adt_dict[yr]

    #  allplat_annual[yr]['adt'] =  allplat_annual[yr].apply(lambda row: adt_data.sel(time=row.datetime, latitude=row.latitude, longitude=row.longitude, method='nearest'), axis=1)
    allplat_annual[yr]['adt'] =  allplat_annual[yr].apply(lambda row: get_adt_for_obs(row), axis=1)
    allplat_annual[yr]['sla'] =  allplat_annual[yr].apply(lambda row: get_sla_for_obs(row), axis=1)



/var/folders/nt/sjynqxjj7cz9r15fkd5d4r_40000gp/T/ipykernel_94730/2005159668.py:19: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  allplat_annual[yr]['adt'] =  allplat_annual[yr].apply(lambda row: get_adt_for_obs(row), axis=1)
/var/folders/nt/sjynqxjj7cz9r15fkd5d4r_40000gp/T/ipykernel_94730/2005159668.py:20: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  allplat_annual[yr]['sla'] =  allplat_annual[yr].apply(lambda row: get_sla_for_obs(row), axis=1)
/var/folders/nt/sjynqxjj7cz9r15fkd5d4r_40000gp/T/ipykernel_947

In [41]:
allplat_added = pd.concat(allplat_annual.values(), axis=0)
allplat_added

,profid,wmoid,prof_datetag,cluster,datetime,latitude,longitude,linear_time,ydcos,ydsin,...,sampleid,cruiseid,nearest_profid,atm_pres_hPa,fco2rec,year,month,day,adt,sla
125,5904187_id002,5904187.0,5904187_20140424,8.0,2014-04-24 01:35:59.999999,-45.027000,209.980000,113.066667,-0.365551,0.930791,...,NaN,NaN,NaN,NaN,NaN,2014,4,24,0.4299,0.1264
126,5904187_id003,5904187.0,5904187_20140429,8.0,2014-04-29 10:21:59.999998,-44.925000,209.465000,118.431944,-0.449781,0.893139,...,NaN,NaN,NaN,NaN,NaN,2014,4,29,0.4306,0.1146
127,5904187_id004,5904187.0,5904187_20140504,8.0,2014-05-04 17:40:00.000001,-45.037000,209.571000,123.736111,-0.529291,0.848440,...,NaN,NaN,NaN,NaN,NaN,2014,5,4,0.4329,0.1294
128,5904187_id005,5904187.0,5904187_20140510,8.0,2014-05-10 00:45:59.999996,-44.769000,209.377000,129.031944,-0.604283,0.796770,...,NaN,NaN,NaN,NaN,NaN,2014,5,10,0.4693,0.1386
129,5904187_id006,5904187.0,5904187_20140515,8.0,2014-05-15 07:09:00.000005,-44.989000,209.152000,134.297917,-0.673884,0.738837,...,NaN,NaN,NaN,NaN,NaN,2014,5,15,0.4377,0.1217
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6083,NaN,NaN,NaN,8.0,2023-03-22 12:21:59,-45.859585,-116.013950,3367.515266,0.188897,0.981997,...,69920230226_id4,69920230226,5905351_id198,100580.0,424.277,2023,3,22,0.5368,0.1639
6084,NaN,NaN,NaN,8.0,2023-03-23 12:21:59,-44.966725,-105.703550,3368.515266,0.171978,0.985101,...,69920230226_id4,69920230226,3902173_id124,100500.0,436.748,2023,3,23,0.6402,0.2362
6085,NaN,NaN,NaN,8.0,2023-03-24 04:40:29,-46.058740,-97.534350,3369.194780,0.160451,0.987044,...,69920230226_id4,69920230226,3901295_id227,100910.0,439.133,2023,3,24,0.5242,0.1762
6086,NaN,NaN,NaN,8.0,2023-01-29 11:36:30,-43.797390,52.461065,3315.483681,0.884354,0.466817,...,35MV20230123_id0,35MV20230123,7900496_id222,101900.0,404.716,2023,1,29,-0.3071,-0.0191


In [42]:
shipDF_added = allplat_added[allplat_added.wmoid.isna()]

floatDF_added = allplat_added[~allplat_added.wmoid.isna()]
floatDF_added['wmoid'] = floatDF_added['wmoid'].astype(int)

/var/folders/nt/sjynqxjj7cz9r15fkd5d4r_40000gp/T/ipykernel_94730/2832873562.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  floatDF_added['wmoid'] = floatDF_added['wmoid'].astype(int)


In [14]:
allplat_added[['adt', 'sla']].describe()

,adt,sla
count,17253.000000,17265.000000
mean,-0.132991,0.077988
std,0.738165,0.095755
min,-1.601300,-0.770200
25%,-0.847100,0.030400
50%,0.018200,0.074700
75%,0.557500,0.126000
max,1.619400,1.020700


# Save

In [35]:
from datetime import datetime
# datetag = datetime.now().strftime('_%Y%m%d') 

In [17]:

# floatDF_added['cluster'] = floatDF_added['cluster'].astype(int)
shipDF_added.columns

Index(['profid', 'wmoid', 'prof_datetag', 'cluster', 'datetime', 'latitude',
       'longitude', 'linear_time', 'ydcos', 'ydsin', 'decimalyr', 'CT', 'SA',
       'mld', 'pCO2_standard', 'pCO2_pHbias3', 'pCO2_pHbias5',
       'pCO2_pHbias3_pK1', 'pco2', 'pco2_atm', 'delta_pco2', 'sampleid',
       'cruiseid', 'nearest_profid', 'atm_pres_hPa', 'fco2rec', 'year',
       'month', 'day', 'adt', 'sla'],
      dtype='object')

In [38]:
floatDF.columns

Index(['profid', 'wmoid', 'prof_datetag', 'cluster', 'datetime', 'latitude',
       'longitude', 'linear_time', 'ydcos', 'ydsin', 'decimalyr', 'CT', 'SA',
       'mld', 'pCO2_standard', 'pCO2_pHbias3', 'pCO2_pHbias5',
       'pCO2_pHbias3_pK1', 'pco2', 'pco2_atm', 'delta_pco2'],
      dtype='object')

In [43]:
use_cols = ['profid', 'wmoid', 'prof_datetag', 'cluster', 
              'datetime', 'latitude', 'longitude', 
              'linear_time', 'ydcos', 'ydsin', 'decimalyr', 
              'CT', 'SA', 'mld', 
              'pCO2_standard', 'pCO2_pHbias3', 'pCO2_pHbias5', 'pCO2_pHbias3_pK1', 'pCO2_pHbias5_pK1', 
              'pco2', 'pco2_atm', 'delta_pco2',
              'year', 'month', 'day', 
              'adt', 'sla']

save = True

datetag = datetime.now().strftime('%Y%m%d') 
filepath = '../working-vars/regression/inputs/'
filename = 'floatDF_ADT_SLA_soccom20m_pco2_pHbias5_pK1_PCM_8pc8class_acc' + datetag + '.csv'
if save:
    floatDF_added[use_cols].to_csv(filepath + filename)
    print('Saved to ' + filepath + filename)

Saved to ../working-vars/regression/inputs/floatDF_ADT_SLA_soccom20m_pco2_pHbias5_pK1_PCM_8pc8class_acc20260119.csv


In [19]:
from datetime import datetime

In [20]:
use_cols = ['sampleid', 'cruiseid', 'nearest_profid', 
              'cluster', 'datetime', 'latitude', 'longitude', 
              'linear_time', 'ydcos', 'ydsin', 'decimalyr', 
              'CT', 'SA', 'mld', 
              'pco2', 'pco2_atm', 'delta_pco2', 
              'atm_pres_hPa', 'fco2rec', 
              'year', 'month', 'day', 
              'adt','sla']

save = True

datetag = datetime.now().strftime('%Y%m%d') 
filepath = '../working-vars/regression/inputs/'
filename = 'shipDF_ADT_SLA_socatv2024_1d_pCO2converted_co2sys_PCM_8pc8class_acc' + datetag + '.csv'
if save:
    shipDF_added[use_cols].to_csv(filepath + filename)
    print('Saved to ' + filepath + filename)

Saved to ../working-vars/regression/inputs/shipDF_ADT_SLA_socatv2024_1d_pCO2converted_co2sys_PCM_8pc8class_acc20260112.csv


In [22]:
shipDF_added[use_cols]

,sampleid,cruiseid,nearest_profid,cluster,datetime,latitude,longitude,linear_time,ydcos,ydsin,...,pco2,pco2_atm,delta_pco2,atm_pres_hPa,fco2rec,year,month,day,adt,sla
0,06AQ20131221_id1,06AQ20131221,6901546_id010,1.0,2014-02-28 11:59:57,-44.270400,14.262600,58.499965,0.534956,0.844880,...,350.623343,393.671629,-43.048286,101780.0,349.3120,2014,2,28,0.2895,0.1811
1,06AQ20131221_id1,06AQ20131221,3900654_id210,1.0,2014-03-01 12:23:57,-42.418600,15.028100,59.516632,0.520098,0.854106,...,344.847836,393.739319,-48.891483,100360.0,343.5760,2014,3,1,0.2908,0.0673
2,06AQ20131221_id1,06AQ20131221,1901394_id151,1.0,2014-03-02 12:21:08,-40.465300,15.811950,60.514676,0.505358,0.862910,...,348.669096,393.814010,-45.144914,101310.0,347.4595,2014,3,2,0.7578,0.2956
3,06AQ20141203_id0,06AQ20141203,6901637_id001,1.0,2014-12-05 12:09:30,-40.512800,10.398900,338.506597,0.896030,-0.443994,...,336.361458,395.882833,-59.521375,101150.0,335.1330,2014,12,5,0.1577,-0.1003
4,06AQ20141203_id0,06AQ20141203,6901639_id001,1.0,2014-12-06 12:23:35,-42.980100,8.505300,339.516377,0.903607,-0.428363,...,373.753244,395.817479,-22.064235,101790.0,372.3090,2014,12,6,0.1662,0.0192
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6083,69920230226_id4,69920230226,5905351_id198,8.0,2023-03-22 12:21:59,-45.859585,-116.013950,3367.515266,0.188897,0.981997,...,425.891924,415.011090,10.880833,100580.0,424.2770,2023,3,22,0.5368,0.1639
6084,69920230226_id4,69920230226,3902173_id124,8.0,2023-03-23 12:21:59,-44.966725,-105.703550,3368.515266,0.171978,0.985101,...,438.396734,415.040189,23.356545,100500.0,436.7480,2023,3,23,0.6402,0.2362
6085,69920230226_id4,69920230226,3901295_id227,8.0,2023-03-24 04:40:29,-46.058740,-97.534350,3369.194780,0.160451,0.987044,...,440.815583,415.004663,25.810920,100910.0,439.1330,2023,3,24,0.5242,0.1762
6086,35MV20230123_id0,35MV20230123,7900496_id222,8.0,2023-01-29 11:36:30,-43.797390,52.461065,3315.483681,0.884354,0.466817,...,406.204779,415.030770,-8.825991,101900.0,404.7160,2023,1,29,-0.3071,-0.0191
